In [2]:
import requests
import pandas as pd
import time
import os

# ==========================================
# [수정1] 브랜드/카테고리 조합 리스트
# ==========================================
BRANDS_CATEGORIES = [
    # {"brand": "filluminate",   "category": "001", "category_name": "상의"},
    # {"brand": "filluminate",   "category": "002", "category_name": "아우터"},
    # {"brand": "filluminate",   "category": "003", "category_name": "하의"},
    {"brand": "jemut",  "category": "001", "category_name": "상의"},  
    {"brand": "jemut",  "category": "002", "category_name": "아우터"},
    {"brand": "jemut",  "category": "003", "category_name": "하의"},
    # {"brand": "travel",  "category": "001", "category_name": "상의"},  
    # {"brand": "travel",  "category": "002", "category_name": "아우터"},
    # {"brand": "travel",  "category": "003", "category_name": "하의"},
]

# [수정2] 4월 날짜 범위
DATE_FROM = "2026-04-01"
DATE_TO   = "2026-04-30"

PAGE_SIZE = 30

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/147.0.0.0 Safari/537.36",
    "Referer": "https://www.musinsa.com/",
    "Origin": "https://www.musinsa.com",
}

session = requests.Session()
session.headers.update(headers)

# ==========================================
# request_json (기존과 동일)
# ==========================================
def request_json(url, params=None, timeout=10, retry=3):
    last_status = None
    last_error = None
    for attempt in range(1, retry + 1):
        try:
            response = session.get(url, params=params, timeout=timeout)
            last_status = response.status_code
            if response.status_code == 200:
                return response.json(), response.status_code
            print(f"요청 실패 ({response.status_code}) -> 재시도 {attempt}/{retry}")
        except Exception as e:
            last_error = e
            print(f"요청 오류 -> 재시도 {attempt}/{retry}: {e}")
        time.sleep(2 * attempt)
    if last_error is not None:
        print(f"최종 요청 오류: {last_error}")
    return None, last_status

# ==========================================
# 만족도 파싱 함수 (기존과 동일)
# ==========================================
def parse_satisfaction(survey):
    if not survey:
        return ""
    questions = survey.get("questions", [])
    parts = []
    for q in questions:
        attribute = q.get("attribute", "")
        answers = q.get("answers", [])
        answer_text = ", ".join([a.get("answerShortText", "") for a in answers])
        parts.append(f"{attribute}: {answer_text}")
    return " / ".join(parts)

# ==========================================
# escape_excel 함수 (기존과 동일)
# ==========================================
def escape_excel(text):
    if isinstance(text, str) and text.startswith(('-', '=', '+', '@', '#')):
        return "'" + text
    return text

# ==========================================
# 리뷰 수집 함수 - [수정3] 날짜 필터 추가 (리뷰타입 컬럼 포함)
# ==========================================
def get_reviews(goods_no, target_count, collected_ids, max_pages=9999):
    reviews = []
    stop_crawling = False

    for page in range(max_pages):
        if len(reviews) >= target_count or stop_crawling:
            break

        url = "https://goods.musinsa.com/api2/review/v1/view/list"
        params = {
            "page":              page,
            "pageSize":          10,
            "goodsNo":           goods_no,
            "sort":              "new",  # 최신순 정렬 (날짜 필터에 필수)
            "selectedSimilarNo": goods_no,
            "myFilter":          "false",
            "hasPhoto":          "false",
            "isExperience":      "false",
        }
        try:
            json_data, status_code = request_json(url, params=params, timeout=10, retry=3)
            if not json_data:
                print(f"  상태코드 {status_code} -> 중단")
                break

            data = json_data.get("data", {})
            review_list = data.get("list", [])

            if not review_list:
                break

            for review in review_list:
                review_no   = str(review.get("no", ""))
                review_date = review.get("createDate", "")

                # ===== [수정3] 날짜 필터 =====
                if review_date > DATE_TO:    # 5월 이후 → 스킵
                    continue
                if review_date < DATE_FROM:  # 3월 이전 → 중단
                    stop_crawling = True
                    break
                # =============================

                # 기존 중복 체크 로직 (그대로 유지)
                if review_no in collected_ids:
                    stop_crawling = True
                    break

                profile = review.get("userProfileInfo") or {}
                images  = review.get("images") or []
                reviews.append({
                    "goodsNo":  str(goods_no),
                    "리뷰번호": review_no,
                    "리뷰타입": review.get("type", ""),   # 기존 코드에 있던 컬럼
                    "작성자":   profile.get("userNickName", ""),
                    "리뷰내용": review.get("content", ""),
                    "평점":     int(review.get("grade", 0)),
                    "체험단":   review.get("type") == "experience",
                    "구매옵션": review.get("goodsOption", ""),
                    "키":       profile.get("userHeight", ""),
                    "몸무게":   profile.get("userWeight", ""),
                    "성별":     profile.get("reviewSex", ""),
                    "작성일":   review_date,
                    "만족도":   parse_satisfaction(review.get("reviewSurveySatisfaction")),
                    "사진유무": len(images) > 0,
                    "도움돼요": review.get("likeCount", 0),
                })

                if len(reviews) >= target_count:
                    break

            total_pages = data.get("page", {}).get("totalPages", 0)
            if page >= total_pages - 1:
                break

            if len(reviews) % 100 == 0 and len(reviews) > 0:
                print(f"  새로운 리뷰 {len(reviews)}개 수집 중...")
            time.sleep(2)

        except Exception as e:
            print(f"  리뷰 수집 오류: {e}")
            time.sleep(2)
            break

    return reviews

# ==========================================
# 상품 통계 수집 함수 (기존과 동일)
# ==========================================
def get_goods_stats(goods_no):
    url = f"https://goods-detail.musinsa.com/api2/goods/{goods_no}/stat"
    try:
        json_data, status_code = request_json(url, timeout=10, retry=3)
        if json_data:
            sales_count = json_data.get("data", {}).get("purchaseTotal", 0)
            views_count = json_data.get("data", {}).get("pageViewTotal", 0)
            return sales_count, views_count
        else:
            print(f"  통계 수집 실패 ({goods_no}): {status_code}")
    except Exception as e:
        print(f"  통계 수집 오류 ({goods_no}): {e}")
    return 0, 0

# ==========================================
# 저장 함수들 - [수정4] CSV 경로를 인자로 받도록 변경
# ==========================================
def save_reviews(reviews, reviews_csv):
    if not reviews:
        return
    df_r = pd.DataFrame(reviews)
    # 줄바꿈 문자 제거 (CSV 깨짐 방지)
    df_r["리뷰내용"] = df_r["리뷰내용"].astype(str).str.replace(r'[\n\r\t]', ' ', regex=True)
    # 엑셀 오류 방지 처리
    for col in ['리뷰내용', '작성자']:
        if col in df_r.columns:
            df_r[col] = df_r[col].apply(escape_excel)
    if os.path.exists(reviews_csv):
        df_r.to_csv(reviews_csv, mode="a", header=False, index=False, encoding="utf-8-sig")
    else:
        df_r.to_csv(reviews_csv, mode="w", header=True, index=False, encoding="utf-8-sig")

def save_error(goods_no, error_type, message, errors_csv):
    df_e = pd.DataFrame([{
        "goodsNo":    goods_no,
        "error_type": error_type,
        "message":    message,
        "time":       pd.Timestamp.now().strftime("%Y-%m-%d %H:%M:%S"),
    }])
    if os.path.exists(errors_csv):
        df_e.to_csv(errors_csv, mode="a", header=False, index=False, encoding="utf-8-sig")
    else:
        df_e.to_csv(errors_csv, mode="w", header=True, index=False, encoding="utf-8-sig")


# ==========================================
# [수정5] 브랜드/카테고리 for문으로 전체 감싸기
# ==========================================
for combo in BRANDS_CATEGORIES:
    BRAND         = combo["brand"]
    CATEGORY      = combo["category"]
    CATEGORY_NAME = combo["category_name"]

    # [수정6] 파일명에 _2026-04 추가 (기존 파일과 분리)
    PRODUCTS_CSV = f"{BRAND}_products_{CATEGORY}_2026-04.csv"
    REVIEWS_CSV  = f"{BRAND}_reviews_{CATEGORY}_2026-04.csv"
    ERRORS_CSV   = f"{BRAND}_errors_{CATEGORY}_2026-04.csv"

    # Referer 업데이트
    session.headers["Referer"] = f"https://www.musinsa.com/brand/{BRAND}"

    print(f"\n{'='*60}")
    print(f"▶ 브랜드: {BRAND} | 카테고리: {CATEGORY_NAME} ({CATEGORY}) | 기간: {DATE_FROM} ~ {DATE_TO}")
    print(f"{'='*60}")

    # ==========================================
    # 브랜드 전체 상품 수집 (기존과 동일)
    # ==========================================
    all_data = []
    page = 1
    next_page_url = None

    while True:
        try:
            if next_page_url:
                json_data, status_code = request_json(next_page_url, timeout=10, retry=3)
            else:
                url = "https://api.musinsa.com/api2/dp/v2/plp/goods"
                params = {
                    "gf":        "A",
                    "isSoldOut": "true",
                    "sortCode":  "REVIEW",
                    "category":  CATEGORY,
                    "brand":     BRAND,
                    "size":      PAGE_SIZE,
                    "caller":    "FLAGSHIP",
                    "page":      page,
                }
                json_data, status_code = request_json(url, params=params, timeout=10, retry=3)

            if not json_data:
                print(f"상품 수집 실패 (page {page}): {status_code}")
                break

            data = json_data.get("data", {})
            goods_list = data.get("list", [])
            if not goods_list:
                break

            for item in goods_list:
                is_sold_out = item.get("isSoldOut", False)
                all_data.append({
                    "플랫폼":   "무신사",
                    "카테고리": CATEGORY_NAME,
                    "브랜드":   item.get("brandName", ""),
                    "goodsNo":  str(item.get("goodsNo", "")),
                    "상품명":   item.get("goodsName", ""),
                    "정가":     item.get("normalPrice", 0),
                    "판매가":   item.get("price", 0),
                    "할인율":   item.get("saleRate", 0),
                    "리뷰수":   item.get("reviewCount", 0),
                    "리뷰점수": item.get("reviewScore", 0),
                    "판매상태": "SOLD_OUT" if is_sold_out else "SALE",  # 기존 코드에 있던 컬럼
                })

            pagination  = data.get("pagination", {})
            total_pages = pagination.get("totalPages", 1)
            print(f"상품 수집 중... {page}/{total_pages} 페이지 (누적 {len(all_data)}개)")

            next_page_url = pagination.get("nextPageUrl")

            if not pagination.get("hasNext", False) or page >= total_pages:
                break
            page += 1
            time.sleep(2)

        except Exception as e:
            print(f"상품 수집 오류 (page {page}): {e}")
            time.sleep(2)
            break

    df = pd.DataFrame(all_data)
    df = df.sort_values("리뷰수", ascending=False).reset_index(drop=True)
    df["조회수"]    = 0
    df["누적판매수"] = 0
    print(f"\n상품 총 {len(df)}개 수집 완료 (리뷰 많은 순)")
    print(df[["카테고리", "브랜드", "상품명", "리뷰수"]].to_string(index=False))

    # ==========================================
    # 재시작 로직 (기존과 동일)
    # ==========================================
    collected_review_ids = set()

    if os.path.exists(PRODUCTS_CSV):
        df_existing = pd.read_csv(PRODUCTS_CSV)
        df_existing["goodsNo"] = df_existing["goodsNo"].astype(str)
        df_existing.set_index("goodsNo", inplace=True)
        print(f"\n기존 수집된 상품 {len(df_existing)}개 확인 -> 기존 정보 최신화 및 신규 리뷰 수집 예정")
    else:
        df_existing = pd.DataFrame()
        print("\n기존 수집 상품 파일 없음 -> 처음부터 수집 시작")

    if os.path.exists(REVIEWS_CSV):
        df_existing_reviews = pd.read_csv(REVIEWS_CSV)
        collected_review_ids = set(df_existing_reviews["리뷰번호"].astype(str).tolist())
        print(f"기존 수집된 개별 리뷰 {len(collected_review_ids)}개 확인 -> 중복 리뷰 스킵 예정")

    # ==========================================
    # 전체 상품 리뷰 및 통계 수집 (기존과 동일, save 함수만 인자 추가)
    # ==========================================
    total_items = len(df)

    for idx, row in df.iterrows():
        goods_no     = row["goodsNo"]
        target_count = int(row["리뷰수"])
        current_item = idx + 1

        is_new_product = df_existing.empty or goods_no not in df_existing.index
        print(f"\n[{current_item}/{total_items}] 처리 중: {str(row['상품명'])[:25]} (총 리뷰 수: {target_count}개)")

        # 1. 통계 수집
        try:
            sales, views = get_goods_stats(goods_no)
        except Exception as e:
            print(f"  통계 수집 실패: {e}")
            save_error(goods_no, "stats", str(e), ERRORS_CSV)
            if not is_new_product:
                sales = df_existing.at[goods_no, "누적판매수"]
                views = df_existing.at[goods_no, "조회수"]
            else:
                sales, views = 0, 0
        time.sleep(2)

        # 3. df_existing 최신 정보 업데이트
        updated_info = {
            "플랫폼":    row["플랫폼"],
            "카테고리":  row["카테고리"],
            "브랜드":    row["브랜드"],
            "상품명":    row["상품명"],
            "정가":      row["정가"],
            "판매가":    row["판매가"],
            "할인율":    row["할인율"],
            "조회수":    views,
            "누적판매수": sales,
            "리뷰수":    row["리뷰수"],
            "리뷰점수":  row["리뷰점수"],
            "판매상태":  row["판매상태"],  # 기존 코드에 있던 컬럼
        }
        for key, value in updated_info.items():
            df_existing.at[goods_no, key] = value

        if is_new_product:
            print(f"  -> 신규 상품 저장 완료 (누적판매: {sales} | 조회수: {views})")
        else:
            print(f"  -> 기존 상품 정보 최신화 완료 (누적판매: {sales} | 조회수: {views})")

        # 4. 리뷰 수집 (날짜 필터는 get_reviews 내부에서 처리)
        if not goods_no or target_count == 0:
            print(f"  [스킵] 리뷰 없음")
        else:
            print(f"  새로운 4월 리뷰 수집 확인 중...")
            try:
                reviews = get_reviews(goods_no, target_count=target_count, collected_ids=collected_review_ids)
                if len(reviews) > 0:
                    save_reviews(reviews, REVIEWS_CSV)
                    collected_review_ids.update([str(r["리뷰번호"]) for r in reviews])
                    print(f"  -> 새로운 4월 리뷰 {len(reviews)}개 수집 및 추가 완료!")
                else:
                    print(f"  -> 4월 리뷰가 없습니다.")
            except Exception as e:
                print(f"  리뷰 수집 실패: {e}")
                save_error(goods_no, "review", str(e), ERRORS_CSV)

        time.sleep(2)

    # ==========================================
    # 최종 상품 데이터 CSV 저장 (기존과 동일)
    # ==========================================
    df_existing.reset_index(inplace=True)
    if 'index' in df_existing.columns:
        df_existing.rename(columns={'index': 'goodsNo'}, inplace=True)
    df_existing.to_csv(PRODUCTS_CSV, index=False, encoding="utf-8-sig")
    print(f"\n[{BRAND}/{CATEGORY_NAME}] 모든 상품 정보 최신화 및 CSV 저장 완료!")

print(f"\n{'='*60}")
print("전체 브랜드/카테고리 수집 완료!")
print(f"{'='*60}")


▶ 브랜드: jemut | 카테고리: 상의 (001) | 기간: 2026-04-01 ~ 2026-04-30
상품 수집 중... 1/37 페이지 (누적 30개)
상품 수집 중... 2/37 페이지 (누적 60개)
상품 수집 중... 3/37 페이지 (누적 90개)
상품 수집 중... 4/37 페이지 (누적 120개)
상품 수집 중... 5/37 페이지 (누적 150개)
상품 수집 중... 6/37 페이지 (누적 180개)
상품 수집 중... 7/37 페이지 (누적 210개)
상품 수집 중... 8/37 페이지 (누적 240개)
상품 수집 중... 9/37 페이지 (누적 270개)
상품 수집 중... 10/37 페이지 (누적 300개)
상품 수집 중... 11/37 페이지 (누적 330개)
상품 수집 중... 12/37 페이지 (누적 360개)
상품 수집 중... 13/37 페이지 (누적 390개)
상품 수집 중... 14/37 페이지 (누적 420개)
상품 수집 중... 15/37 페이지 (누적 450개)
상품 수집 중... 16/37 페이지 (누적 480개)
상품 수집 중... 17/37 페이지 (누적 510개)
상품 수집 중... 18/37 페이지 (누적 540개)
상품 수집 중... 19/37 페이지 (누적 570개)
상품 수집 중... 20/37 페이지 (누적 600개)
상품 수집 중... 21/37 페이지 (누적 630개)
상품 수집 중... 22/37 페이지 (누적 660개)
상품 수집 중... 23/37 페이지 (누적 690개)
상품 수집 중... 24/37 페이지 (누적 720개)
상품 수집 중... 25/37 페이지 (누적 750개)
상품 수집 중... 26/37 페이지 (누적 780개)
상품 수집 중... 27/37 페이지 (누적 810개)
상품 수집 중... 28/37 페이지 (누적 840개)
상품 수집 중... 29/37 페이지 (누적 870개)
상품 수집 중... 30/37 페이지 (누적 900개)
상품 수집 중... 31/37 페이지 